# 04 - Data Integration for Pre-training

This notebook combines WoSIS, ERA5, and SoilGrids data into training sequences.

## Objectives
- Merge data sources by location and time
- Handle missing values and quality control
- Create training-ready sequences
- Implement spatial train/validation/test splits

## Output
- `data/processed/training_sequences.parquet`

In [ ]:
# Standard imports
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

# Project imports
from data.data_processor import DataProcessor, ProcessorConfig

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')

RAW_DIR = Path('../../data/raw')
PROCESSED_DIR = Path('../../data/processed')
RESULTS_DIR = Path('../../results/data_analysis')

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Individual Datasets

In [ ]:
# Load WoSIS profiles
wosis_path = RAW_DIR / 'wosis' / 'wosis_snapshot.parquet'

if wosis_path.exists():
    wosis_df = pd.read_parquet(wosis_path)
    print(f"Loaded {len(wosis_df):,} WoSIS profiles")
    print(f"Columns: {wosis_df.columns.tolist()}")
else:
    print(f"WoSIS data not found at {wosis_path}")
    wosis_df = None

In [ ]:
# Load ERA5 summaries (if pre-computed)
era5_path = RAW_DIR / 'era5' / 'era5_daily.nc'

if era5_path.exists():
    print(f"ERA5 data available at {era5_path}")
    # Will be loaded during sequence creation
else:
    print(f"ERA5 data not found at {era5_path}")

In [ ]:
# Check SoilGrids availability
soilgrids_dir = RAW_DIR / 'soilgrids'

if soilgrids_dir.exists():
    tif_count = len(list(soilgrids_dir.glob('*.tif')))
    print(f"SoilGrids: {tif_count} GeoTIFF files available")
else:
    print(f"SoilGrids data not found at {soilgrids_dir}")

## 2. Data Quality Checks

In [ ]:
# Coordinate validation
if wosis_df is not None:
    # Check coordinate ranges
    print("Coordinate validation:")
    print(f"  Latitude range: [{wosis_df['latitude'].min():.2f}, {wosis_df['latitude'].max():.2f}]")
    print(f"  Longitude range: [{wosis_df['longitude'].min():.2f}, {wosis_df['longitude'].max():.2f}]")
    
    # Check for invalid coordinates
    invalid_lat = (wosis_df['latitude'] < -90) | (wosis_df['latitude'] > 90)
    invalid_lon = (wosis_df['longitude'] < -180) | (wosis_df['longitude'] > 180)
    
    print(f"\nInvalid coordinates:")
    print(f"  Invalid latitude: {invalid_lat.sum()}")
    print(f"  Invalid longitude: {invalid_lon.sum()}")
    
    # Remove invalid
    if invalid_lat.sum() > 0 or invalid_lon.sum() > 0:
        wosis_df = wosis_df[~invalid_lat & ~invalid_lon]
        print(f"\nAfter cleaning: {len(wosis_df):,} profiles")

In [ ]:
# Target property validation
TARGET_PROPERTIES = ['SOC', 'pH', 'clay', 'sand', 'silt', 'N', 'CEC']

if wosis_df is not None:
    available_targets = [p for p in TARGET_PROPERTIES if p in wosis_df.columns]
    
    print("Target property statistics:")
    print("-" * 60)
    
    for prop in available_targets:
        valid = wosis_df[prop].notna()
        print(f"\n{prop}:")
        print(f"  Available: {valid.sum():,} ({100*valid.mean():.1f}%)")
        if valid.sum() > 0:
            print(f"  Range: [{wosis_df.loc[valid, prop].min():.2f}, {wosis_df.loc[valid, prop].max():.2f}]")
            print(f"  Mean: {wosis_df.loc[valid, prop].mean():.2f}")

## 3. Create Training Sequences

In [ ]:
# Initialize data processor
config = ProcessorConfig(
    temporal_window_days=365,
    target_properties=tuple(available_targets) if wosis_df is not None else (),
    spatial_grid_size=5
)

processor = DataProcessor(config)

print("Processor configuration:")
print(f"  Temporal window: {config.temporal_window_days} days")
print(f"  Target properties: {config.target_properties}")
print(f"  Spatial grid size: {config.spatial_grid_size}")

In [ ]:
# Create sequences (if all data available)
output_path = PROCESSED_DIR / 'training_sequences.parquet'

if wosis_df is not None and era5_path.exists() and soilgrids_dir.exists():
    print("Creating training sequences...")
    
    # This would take significant time for full dataset
    # For prototyping, use a sample
    sample_df = wosis_df.sample(min(1000, len(wosis_df)), random_state=42)
    
    sequences_df = processor.create_training_sequences(
        wosis_df=sample_df,
        era5_path=era5_path,
        soilgrids_dir=soilgrids_dir,
        output_path=output_path
    )
    
    print(f"\nCreated {len(sequences_df):,} training sequences")
else:
    print("Cannot create sequences - missing data sources")
    print("Ensure WoSIS, ERA5, and SoilGrids data are downloaded")
    sequences_df = None

## 4. Train/Validation/Test Split

In [ ]:
# Spatial-aware split
if sequences_df is not None:
    train_df, val_df, test_df = processor.split_data(
        sequences_df,
        test_size=0.15,
        val_size=0.10,
        spatial_aware=True
    )
    
    print("Data splits:")
    print(f"  Train: {len(train_df):,} ({100*len(train_df)/len(sequences_df):.1f}%)")
    print(f"  Validation: {len(val_df):,} ({100*len(val_df)/len(sequences_df):.1f}%)")
    print(f"  Test: {len(test_df):,} ({100*len(test_df)/len(sequences_df):.1f}%)")

In [ ]:
# Visualize spatial split
if sequences_df is not None and 'spatial_group' in sequences_df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for ax, (name, df) in zip(axes, [('Train', train_df), ('Validation', val_df), ('Test', test_df)]):
        ax.scatter(df['longitude'], df['latitude'], s=5, alpha=0.5)
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.set_title(f'{name} Set (n={len(df):,})')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'spatial_split.png', dpi=150)
    plt.show()

## 5. Feature Normalization

In [ ]:
# Normalize features
if sequences_df is not None:
    # Identify numeric columns
    exclude_cols = ['profile_id', 'latitude', 'longitude', 'date', 
                    'weather_sequence', 'spatial_group']
    numeric_cols = [c for c in sequences_df.select_dtypes(include=[np.number]).columns
                   if c not in exclude_cols]
    
    print(f"Normalizing {len(numeric_cols)} features:")
    for col in numeric_cols[:10]:
        print(f"  - {col}")
    if len(numeric_cols) > 10:
        print(f"  ... and {len(numeric_cols) - 10} more")
    
    # Normalize on train set only
    train_norm, stats = processor.normalize_features(train_df, numeric_cols)
    
    # Apply same normalization to val/test
    val_norm = val_df.copy()
    test_norm = test_df.copy()
    
    for col in numeric_cols:
        mean, std = stats[col]
        if std > 0:
            val_norm[col] = (val_df[col] - mean) / std
            test_norm[col] = (test_df[col] - mean) / std
    
    # Save normalization stats
    with open(PROCESSED_DIR / 'normalization_stats.json', 'w') as f:
        json.dump({k: list(v) for k, v in stats.items()}, f, indent=2)
    
    print("\nNormalization stats saved")

## 6. Pre-training Mask Creation

In [ ]:
# Create masked targets for pre-training
if sequences_df is not None:
    masked_train = processor.create_masking_targets(train_norm, mask_ratio=0.15)
    
    print("Pre-training mask statistics:")
    print(f"  Mask ratio: 15%")
    
    # Analyze mask distribution
    mask_array = np.array(masked_train['mask'].tolist())
    mask_per_property = mask_array.mean(axis=0)
    
    print("\nMask rate per property:")
    for i, prop in enumerate(config.target_properties):
        if i < len(mask_per_property):
            print(f"  {prop}: {100*mask_per_property[i]:.1f}%")

## 7. Save Processed Data

In [ ]:
# Save all splits
if sequences_df is not None:
    # Save raw sequences
    sequences_df.to_parquet(PROCESSED_DIR / 'training_sequences.parquet')
    
    # Save splits
    train_norm.to_parquet(PROCESSED_DIR / 'train.parquet')
    val_norm.to_parquet(PROCESSED_DIR / 'validation.parquet')
    test_norm.to_parquet(PROCESSED_DIR / 'test.parquet')
    
    # Save masked training data
    masked_train.to_parquet(PROCESSED_DIR / 'train_masked.parquet')
    
    print("Saved processed data:")
    for f in PROCESSED_DIR.glob('*.parquet'):
        print(f"  - {f.name}")

## 8. Summary

In [ ]:
# Generate integration summary
summary = {
    'data_sources': {
        'wosis': str(wosis_path),
        'era5': str(era5_path),
        'soilgrids': str(soilgrids_dir)
    },
    'sequences_created': len(sequences_df) if sequences_df is not None else 0,
    'splits': {
        'train': len(train_df) if sequences_df is not None else 0,
        'validation': len(val_df) if sequences_df is not None else 0,
        'test': len(test_df) if sequences_df is not None else 0
    },
    'features': {
        'target_properties': list(config.target_properties),
        'temporal_window': config.temporal_window_days,
        'era5_variables': list(config.era5_variables),
        'soilgrids_layers': list(config.soilgrids_layers)
    },
    'output_files': [
        'training_sequences.parquet',
        'train.parquet',
        'validation.parquet',
        'test.parquet',
        'normalization_stats.json'
    ]
}

print("=" * 50)
print("DATA INTEGRATION SUMMARY")
print("=" * 50)
print(f"\nTotal sequences: {summary['sequences_created']:,}")
print(f"\nSplits:")
for split, count in summary['splits'].items():
    print(f"  {split}: {count:,}")
print(f"\nTarget properties: {len(summary['features']['target_properties'])}")
print(f"Temporal window: {summary['features']['temporal_window']} days")

In [ ]:
# Save summary
with open(RESULTS_DIR / 'integration_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"Summary saved to {RESULTS_DIR / 'integration_summary.json'}")